<a href="https://colab.research.google.com/github/claudiodanielpc/medium/blob/main/portal_datos_abiertos_cdmx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from matplotlib import pyplot as plt
import matplotlib.dates as mdates
from dateutil.rrule import rrule, MONTHLY
from matplotlib.ticker import FixedLocator
from math import floor

In [2]:
headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36'}
##Url básica del portal de datos abiertos
url_basica="https://datos.cdmx.gob.mx/api/3/action/"

In [3]:
#Nombre de los conjuntos de datos
url_datos=url_basica+"package_list"

nom_conjunto=requests.get(url_datos, headers=headers)
nom_conjunto=nom_conjunto.json()
nom_conjunto=nom_conjunto['result']
print("Total de conjuntos de datos:", len(nom_conjunto))

Total de conjuntos de datos: 475


In [4]:
##Obtener datos generales de cada conjunto de datos
titulo = []
organizacion= []
descripcion = []
frecuencia=[]
metadata_creado=[]
metadata_modificado=[]
url_ultimo_archivo=[]


#Buscar
for conjunto in nom_conjunto:
    url_datos=url_basica+"package_show?id="+conjunto
    r=requests.get(url_datos, headers=headers)
    r=r.json()
    titulo.append(r['result']['title'])
    descripcion.append(r['result']['notes'])
    organizacion.append(r['result']['organization']['title'])
    recursos = r['result'].get('resources', [])
    if recursos:
      freq = recursos[0].get('update_frequency', None)
      original_url = recursos[0].get('original_url', None)
    else:
      freq = None
      original_url = None
    frecuencia.append(freq)
    url_ultimo_archivo.append(original_url)
    metadata_creado.append(r['result'].get('metadata_created'))
    metadata_modificado.append(r['result'].get('metadata_modified'))

In [5]:
tabla_conjunto = pd.DataFrame({
    "nombre": titulo,
    "organizacion": organizacion,
    "descripcion": descripcion,
    "frecuencia": frecuencia,
    "metadata_creado": metadata_creado,
    "metadata_modificado": metadata_modificado,
    "url_ultimo_archivo": url_ultimo_archivo
})

tabla_conjunto = tabla_conjunto.apply(lambda x: x.astype(str).str.lower())
#Eliminar acentos
tabla_conjunto = tabla_conjunto.apply(lambda x: x.str.replace('á', 'a').str.replace('é', 'e').str.replace('í', 'i').str.replace('ó', 'o').str.replace('ú', 'u'))
#Eliminar espacios adelante y atrás
tabla_conjunto = tabla_conjunto.apply(lambda x: x.str.strip())

#Variables fecha_creacion y fecha_ult_modif en formato fecha
tabla_conjunto['metadata_creado'] = pd.to_datetime(tabla_conjunto['metadata_creado'], format='%Y-%m-%dT%H:%M:%S.%f')

tabla_conjunto['metadata_modificado'] = pd.to_datetime(tabla_conjunto['metadata_modificado'], format='%Y-%m-%dT%H:%M:%S.%f')
print("Total de conjuntos extraídos:", len(tabla_conjunto))
tabla_conjunto

Total de conjuntos extraídos: 475


,nombre,organizacion,descripcion,frecuencia,metadata_creado,metadata_modificado,url_ultimo_archivo
0,solicitudes *0311,agencia digital de innovacion publica (adip),**descripcion**\r\n\r\nel conjunto de datos de...,mensual,2022-10-05 21:53:24.039621,2024-04-01 20:13:31.531809,https://datos.cdmx.gob.mx/dataset/529aac27-d1c...
1,accidentes terrestres en las alcaldias,instituto de planeacion democratica y prospectiva,accidentes terrestres en las alcaldias,none,2022-01-27 20:03:18.319830,2023-02-15 00:58:46.488348,none
2,acciones sociales vigentes en la ciudad de mexico,consejo de evaluacion del desarrollo social de...,la **politica de desarrollo social** en la ciu...,,2021-01-13 04:21:39.740771,2023-02-15 00:57:03.472691,none
3,actas de defuncion en el registro civil de la ...,direccion general del registro civil,este conjunto de datos tiene como objetivo rep...,trimestral,2021-01-13 04:36:48.558579,2024-01-09 17:35:21.395624,none
4,acueducto planta chapultepec,instituto de planeacion democratica y prospectiva,trazo del acueducto planta chapultepec,none,2022-02-01 21:59:15.157067,2023-02-15 00:59:05.471381,https://datos.cdmx.gob.mx/dataset/09d5e844-3eb...
...,...,...,...,...,...,...,...
470,zonas de patrullaje (cuadrantes) 2018,secretaria de seguridad ciudadana (ssc),este es el directorio de zonas de patrullaje v...,,2021-01-13 04:07:19.450907,2023-07-07 18:17:27.362877,none
471,zonas para impulsar la vivienda para trabajado...,instituto de planeacion democratica y prospectiva,zonas para impulsar y facilitar la construccio...,none,2022-02-01 18:20:17.325613,2023-02-15 00:57:56.178684,none
472,municipios de las zonas metropolitanas en la z...,instituto de planeacion democratica y prospectiva,delimitacion de los municipios de las zonas me...,none,2022-02-02 19:42:13.487964,2023-02-15 00:56:50.546012,none
473,zonas de monumentos historicos,instituto de planeacion democratica y prospectiva,poligono de las zonas declaradas patrimonio po...,none,2022-02-02 19:45:02.253397,2023-02-15 00:58:01.413896,https://datos.cdmx.gob.mx/dataset/b3bdbaef-ced...


In [6]:
tabla_conjunto["frecuencia"].unique()

array(['mensual', 'none', '', 'trimestral', 'historico', 'semestral',
       'anual', 'bimestral', 'semanal'], dtype=object)

In [8]:
pd.DataFrame({
    "frecuencia": tabla_conjunto["frecuencia"].value_counts(dropna=False).index,
    "numero": tabla_conjunto["frecuencia"].value_counts(dropna=False).values,
    "porcentaje": tabla_conjunto["frecuencia"].value_counts(normalize=True, dropna=False).mul(100).round(2).values
}).sort_values("numero", ascending=False)


,frecuencia,numero,porcentaje
0,none,222,46.74
1,,121,25.47
2,historico,43,9.05
3,anual,36,7.58
4,mensual,29,6.11
5,trimestral,15,3.16
6,semestral,6,1.26
7,bimestral,2,0.42
8,semanal,1,0.21


In [9]:
pd.DataFrame({
    "frecuencia": tabla_conjunto["organizacion"].value_counts(dropna=False).index,
    "numero": tabla_conjunto["organizacion"].value_counts(dropna=False).values,
    "porcentaje": tabla_conjunto["organizacion"].value_counts(normalize=True, dropna=False).mul(100).round(2).values
}).sort_values("numero", ascending=False)

,frecuencia,numero,porcentaje
0,instituto de planeacion democratica y prospectiva,205,43.16
1,agencia digital de innovacion publica (adip),50,10.53
2,secretaria de movilidad (semovi),34,7.16
3,secretaria de gestion integral de riesgos y pr...,22,4.63
4,secretaria de administracion y finanzas (saf),21,4.42
5,secretaria del medio ambiente (sedema),20,4.21
6,instituto nacional de estadistica y geografia ...,17,3.58
7,secretaria de salud,15,3.16
8,consejo de evaluacion del desarrollo social de...,15,3.16
9,secretaria de seguridad ciudadana (ssc),10,2.11


In [10]:
pd.DataFrame({
    "frecuencia": tabla_conjunto[tabla_conjunto["organizacion"] ==
                                 "instituto de planeacion democratica y prospectiva"]["frecuencia"]
                    .value_counts(dropna=False).index,

    "numero": tabla_conjunto[tabla_conjunto["organizacion"] ==
                              "instituto de planeacion democratica y prospectiva"]["frecuencia"]
                    .value_counts(dropna=False).values,

    "porcentaje": tabla_conjunto[tabla_conjunto["organizacion"] ==
                                 "instituto de planeacion democratica y prospectiva"]["frecuencia"]
                    .value_counts(normalize=True, dropna=False).mul(100).round(2).values
}).sort_values("numero", ascending=False)


,frecuencia,numero,porcentaje
0,none,199,97.07
1,,3,1.46
2,trimestral,2,0.98
3,historico,1,0.49


In [7]:
hoy = pd.Timestamp.today()
fin_2025 = pd.Timestamp("2025-12-31")

df = tabla_conjunto.copy()

df["metadata_modificado"] = pd.to_datetime(df["metadata_modificado"], errors="coerce")

frecuencia_dias = {
    "mensual": 30,
    "trimestral": 90,
    "semestral": 182,
    "anual": 365,
    "bimestral": 60,
    "semanal": 7,
    "histórico": None,
    "none": None
}

df["dias_esperados"] = df["frecuencia"].map(frecuencia_dias)

# --- Número de actualizaciones faltantes ---
df["num_actualizaciones_faltantes"] = df.apply(
    lambda row: (
        floor((hoy - row["metadata_modificado"]).days / row["dias_esperados"])
        if pd.notna(row["metadata_modificado"]) and pd.notna(row["dias_esperados"])
        else 0
    ),
    axis=1
)

df[
    df["frecuencia"].isin(["mensual", "trimestral", "semestral", "anual", "bimestral", "semanal"])
][[
    "nombre",
    "frecuencia",
    "metadata_modificado",
    "num_actualizaciones_faltantes"
]].sort_values("num_actualizaciones_faltantes", ascending=False).head(40)


,nombre,frecuencia,metadata_modificado,num_actualizaciones_faltantes
394,servicios integrales de locatel,semanal,2023-05-30 19:02:40.476607,131
124,datos de bicicletas (ecobici),mensual,2023-05-25 22:23:21.777969,30
271,matrimonios registrados en la ciudad de mexico,mensual,2023-07-07 18:31:41.771040,29
374,red de meteorologia y radiacion solar (redmet),mensual,2023-06-29 00:44:44.307060,29
370,red automatica de monitoreo atmosferico (rama),mensual,2023-06-28 22:27:25.210881,29
281,nacimientos registrados en la ciudad de mexico,mensual,2023-07-07 18:37:22.504725,29
371,red de recarga externa de la tarjeta de movili...,mensual,2023-07-06 22:08:47.293664,29
244,ingresos del sistema de transporte colectivo m...,mensual,2023-09-02 02:53:05.782118,27
106,convocatorias de licitaciones publicas y anunc...,mensual,2023-12-04 22:11:41.449975,24
379,remuneraciones al personal de la ciudad de mexico,mensual,2024-02-08 19:53:49.830072,22
